# 01b: MOLASS Runs — Shimizu 20260305 Dataset

**Experiment**: 01 — Averaging order effect on MOLASS decomposition  
**Purpose**: Run MOLASS (trim → baseline-correct → quick decomposition) on all 6 datasets with identical default settings  
**Date**: March 2026  
**Follows**: `01a_data_exploration.ipynb` (data QC passed for all 6 datasets)  
**Feeds into**: `01c_comparison_analysis.ipynb`

## What we are doing
- Load each dataset fresh and run the canonical MOLASS pipeline
- Use **identical default settings** for all 6 — no manual tweaking
- Record for each dataset: number of components found, Rg per component, elution proportions
- Save decomposition results to `01b_results.pkl` for use in 01c

## Datasets
| Name   | Sample     | UV λ   | Type                     |
|--------|------------|--------|--------------------------|
| Apo    | Apo protein | 280 nm | Original                 |
| Apo2   | Apo protein | 280 nm | Pre-averaged (19 frames) |
| ATP    | ATP-bound   | 290 nm | Original                 |
| ATP2   | ATP-bound   | 290 nm | Pre-averaged (19 frames) |
| MY     | MY          | 290 nm | Original                 |
| MY2    | MY          | 290 nm | Pre-averaged (19 frames) |

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from molass.DataObjects import SecSaxsData as SSD
import molass; assert molass.get_version() >= "0.8.3", \
    f"molass >= 0.8.3 required, found {molass.get_version()}"
print("molass", molass.get_version())

# ── Set your local data root here ──────────────────────────────────────────
DATA_ROOT = r"C:\Users\takahashi\Dropbox\MOLASS\DATA\20260305"
# ───────────────────────────────────────────────────────────────────────────

DATASETS = {
    "Apo":  {"folder": "Apo",  "uv_signal": 280, "pair": "Apo2"},
    "Apo2": {"folder": "Apo2", "uv_signal": 280, "pair": "Apo"},
    "ATP":  {"folder": "ATP",  "uv_signal": 290, "pair": "ATP2"},
    "ATP2": {"folder": "ATP2", "uv_signal": 290, "pair": "ATP"},
    "MY":   {"folder": "MY",   "uv_signal": 290, "pair": "MY2"},
    "MY2":  {"folder": "MY2",  "uv_signal": 290, "pair": "MY"},
}
SAMPLE_PAIRS = [("Apo", "Apo2", 280), ("ATP", "ATP2", 290), ("MY", "MY2", 290)]

print("Data root:", DATA_ROOT)
print("Exists:", os.path.isdir(DATA_ROOT))


## Run MOLASS Pipeline on All 6 Datasets

Pipeline: `SSD(folder)` → `trimmed_copy()` → `corrected_copy()` → `quick_decomposition()`

Default settings are used throughout — no manual parameter tuning.

In [ ]:
results = {}  # will hold per-dataset results

for name, info in DATASETS.items():
    folder = os.path.join(DATA_ROOT, info["folder"])
    print(f"\n{'='*50}")
    print(f"Processing {name} ...")
    ssd        = SSD(folder)
    trimmed    = ssd.trimmed_copy()
    corrected  = trimmed.corrected_copy()
    decomp     = corrected.quick_decomposition()

    rgs        = decomp.get_rgs()
    props      = decomp.get_proportions()
    n_comp     = decomp.get_num_components()

    print(f"  Components : {n_comp}")
    for i, (rg, prop) in enumerate(zip(rgs, props)):
        print(f"  Component {i+1}: Rg = {rg:.2f} Å,  proportion = {prop:.3f}")

    results[name] = {
        "decomposition": decomp,
        "corrected_ssd": corrected,
        "rgs":           rgs,
        "proportions":   props,
        "num_components": n_comp,
    }

print("\nAll datasets processed.")

## Decomposition Component Plots

For each dataset, plot the decomposed elution components overlaid on the total intensity curve.
Plots are arranged as pairs (original | pre-averaged) for easy comparison.

In [ ]:
for orig, avg, wv_signal in SAMPLE_PAIRS:
    # Note: plot_components() generates its own figure internally;
    # show each dataset separately rather than forcing into subplots.
    print(f"\n{'='*60}")
    print(f"{orig} vs {avg}")
    for name in [orig, avg]:
        if name not in results:
            print(f"  {name}: not loaded")
            continue
        decomp = results[name]["decomposition"]
        label = "Original" if name == orig else "Pre-averaged (19 frames)"
        n = results[name]["num_components"]
        decomp.plot_components(title=f"{name} — {label}  ({n} components)")
        fname = f"01b_decomp_{name.lower()}.png"
        plt.savefig(fname, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"  Saved: {fname}")

## Summary Table

Tabulate the key MOLASS outputs for all 6 datasets.
Pairs are shown adjacent for easy comparison.

In [ ]:
print(f"{'Dataset':<8} {'N_comp':>7} {'Rg_1 (Å)':>10} {'Rg_2 (Å)':>10} {'Prop_1':>8} {'Prop_2':>8}")
print("-" * 55)

for orig, avg, _ in SAMPLE_PAIRS:
    for name in [orig, avg]:
        if name not in results:
            print(f"{name:<8}  (not loaded)")
            continue
        r = results[name]
        rgs   = r["rgs"]
        props = r["proportions"]
        n     = r["num_components"]
        rg1   = f"{rgs[0]:.2f}"   if len(rgs)   > 0 else "—"
        rg2   = f"{rgs[1]:.2f}"   if len(rgs)   > 1 else "—"
        p1    = f"{props[0]:.3f}" if len(props) > 0 else "—"
        p2    = f"{props[1]:.3f}" if len(props) > 1 else "—"
        print(f"{name:<8} {n:>7} {rg1:>10} {rg2:>10} {p1:>8} {p2:>8}")
    print()

## Save Results for 01c

Save the decomposition objects and scalar summaries to `01b_results.pkl`.
The full `decomposition` objects are preserved so 01c can access elution curves and scattering profiles directly.

In [ ]:
save_path = "01b_results.pkl"
with open(save_path, "wb") as f:
    pickle.dump(results, f)
print(f"Saved results to: {save_path}")
print(f"Keys: {list(results.keys())}")

## Observations

*(Filled in after running cells above — March 6, 2026)*

### Number of components (default auto-detection)
- **Apo vs Apo2**: Both → 1 component. Consistent.
- **ATP vs ATP2**: Both → 1 component. Consistent.
- **MY vs MY2**: MY → 2 components; MY2 → **3 components**. ⚠️ Decisive difference.


### Rg values→ Apo/ATP pairs are straightforward to compare (same number of components). MY/MY2 comparison requires deciding on a fixed num_components for fair comparison. Address in `01c_comparison_analysis.ipynb`.

- **Apo**: 33.34 vs 33.40 Å (Δ = +0.06 Å, 0.2%) — essentially identical.### Readiness for comparison

- **ATP**: 32.93 vs 32.82 Å (Δ = −0.11 Å, 0.3%) — essentially identical.

- **MY**: Both components Rg = 32.3 Å (suspicious: two components with identical Rg — likely a decomposition artifact).The original MY decomposition (2 components, both Rg = 32.3 Å) is suspicious: two components with identical Rg suggests the decomposition was fitting the two UV elution peaks (visually different features) as separate UV/XR channels, while the SAXS sees only one species.

- **MY2**: Rg = 97.0, 53.6, 32.4 Å — the main protein peak at 32.4 Å with two large-Rg components that are likely aggregates.

This is an example of pre-averaging **changing the decomposition result decisively**: a qualitatively different number of components with biologically distinct Rg values.

### Key finding: MY vs MY2The MY2 Guinier plot shows components at Rg = 97 and 53.6 Å — plausibly dimer/oligomer aggregates — that were below the S/N threshold in the original MY data.
The pre-averaged MY2 data resolves **three distinct components** where the original MY only found two.